# FLUX/Z-Image Generator - Google Colab

Fast image generation using FLUX models on free T4 GPU.

**Features:**
- Generate from prompts.json or single prompt
- Auto-save to Google Drive
- Progress tracking
- ~15-20 sec per image (much faster than local)

## 1. Setup GPU

In [ ]:
# Check GPU
!nvidia-smi

# Should show Tesla T4 (16GB VRAM) - free tier

## 2. Install Dependencies

In [ ]:
# Install diffusers and dependencies
!pip install -q diffusers transformers accelerate safetensors sentencepiece

import torch
from diffusers import FluxPipeline
from PIL import Image
import json
import os
from datetime import datetime
from google.colab import drive

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Mount Google Drive (for saving images)

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Create output folder
output_base = '/content/drive/MyDrive/flux_images'
os.makedirs(output_base, exist_ok=True)

print(f"Images will be saved to: {output_base}")

## 4. Load FLUX Model

Options:
- `black-forest-labs/FLUX.1-schnell` - Fastest, best for batch generation
- `black-forest-labs/FLUX.1-dev` - Higher quality, slower

In [ ]:
# Load FLUX.1-schnell (fastest)
MODEL_ID = "black-forest-labs/FLUX.1-schnell"

print(f"Loading {MODEL_ID}...")
print("This takes ~2-3 minutes on first run (downloading ~23GB)")

pipe = FluxPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

# Enable memory efficient attention
pipe.enable_model_cpu_offload()

print("Model loaded!")

## 5. Single Image Test

In [ ]:
# Test with a single prompt
test_prompt = "A cute busty Japanese girl, 21 years old, lying on a bed wearing a tiny sheer black lace bra and matching panties. Her large full breasts are visible, deep cleavage showing. She is looking directly at the camera with a seductive, inviting smile. Dark messy hair spread across the pillow. Warm intimate bedroom light, soft shadows. Photorealistic, cinematic lighting."

print("Generating test image...")
image = pipe(
    prompt=test_prompt,
    guidance_scale=0.0,  # schnell works best with 0
    num_inference_steps=4,  # schnell only needs 4 steps!
    max_sequence_length=256,
    height=1280,
    width=720,
).images[0]

# Save test
test_path = f"{output_base}/test_image.png"
image.save(test_path)
print(f"Saved to: {test_path}")

# Display
image

## 6. Batch Generation from prompts.json

In [ ]:
# Upload your prompts.json file
from google.colab import files

print("Upload your prompts.json file:")
uploaded = files.upload()

# Get the filename
prompts_file = list(uploaded.keys())[0]
print(f"Loaded: {prompts_file}")

In [ ]:
# Load prompts
with open(prompts_file, 'r') as f:
    prompts_data = json.load(f)

print(f"Loaded {len(prompts_data)} prompts")
print(f"First prompt: {prompts_data[0]['name']}")

In [ ]:
# Set output folder name (change this for each story)
STORY_NAME = "the_neighbor"  # Change this!

output_folder = f"{output_base}/{STORY_NAME}"
os.makedirs(output_folder, exist_ok=True)

print(f"Output folder: {output_folder}")
print(f"Total images to generate: {len(prompts_data)}")
print(f"Estimated time: {len(prompts_data) * 15 / 60:.1f} minutes")

In [ ]:
# Generate all images
import time

start_time = time.time()
generated = 0
failed = []

for i, item in enumerate(prompts_data):
    name = item['name']
    prompt = item['prompt']
    
    # Add photorealistic style
    full_prompt = f"{prompt} Photorealistic, cinematic lighting, 9:16 vertical portrait."
    
    try:
        print(f"[{i+1}/{len(prompts_data)}] Generating: {name}...", end=" ")
        
        img_start = time.time()
        
        image = pipe(
            prompt=full_prompt,
            guidance_scale=0.0,
            num_inference_steps=4,
            max_sequence_length=256,
            height=1280,
            width=720,
        ).images[0]
        
        # Save
        save_path = f"{output_folder}/{name}.png"
        image.save(save_path)
        
        img_time = time.time() - img_start
        generated += 1
        print(f"✓ ({img_time:.1f}s)")
        
    except Exception as e:
        print(f"✗ Error: {e}")
        failed.append(name)

total_time = time.time() - start_time

print(f"\n{'='*50}")
print(f"Generation complete!")
print(f"Generated: {generated}/{len(prompts_data)}")
print(f"Total time: {total_time/60:.1f} minutes")
print(f"Average per image: {total_time/generated:.1f}s")
if failed:
    print(f"Failed: {failed}")
print(f"Images saved to: {output_folder}")

## 7. Download All Images as ZIP

In [ ]:
# Create ZIP file for easy download
import shutil

zip_path = f"{output_base}/{STORY_NAME}_images"
shutil.make_archive(zip_path, 'zip', output_folder)

print(f"ZIP created: {zip_path}.zip")
print(f"Size: {os.path.getsize(zip_path + '.zip') / 1024 / 1024:.1f} MB")

# Download
files.download(f"{zip_path}.zip")

## 8. Regenerate Failed Images (if any)

In [ ]:
# Retry failed images
if failed:
    print(f"Retrying {len(failed)} failed images...")
    
    for name in failed:
        # Find the prompt
        item = next(p for p in prompts_data if p['name'] == name)
        prompt = item['prompt']
        full_prompt = f"{prompt} Photorealistic, cinematic lighting, 9:16 vertical portrait."
        
        try:
            print(f"Retrying: {name}...", end=" ")
            image = pipe(
                prompt=full_prompt,
                guidance_scale=0.0,
                num_inference_steps=4,
                max_sequence_length=256,
                height=1280,
                width=720,
            ).images[0]
            
            save_path = f"{output_folder}/{name}.png"
            image.save(save_path)
            print("✓")
        except Exception as e:
            print(f"✗ Still failed: {e}")